# RTMDet-tiny reproduction

In [ ]:
import subprocess

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu)
if not gpu.startswith("Tesla T4,"):
    raise RuntimeError("select a Tesla T4 runtime for this reproduction")

In [ ]:
import os
from pathlib import Path

installer = Path("/tmp/uv-install.sh")
subprocess.run(
    [
        "curl",
        "-LsSf",
        "https://astral.sh/uv/0.12.23/install.sh",
        "-o",
        str(installer),
    ],
    check=True,
)
env = os.environ.copy()
env["UV_UNMANAGED_INSTALL"] = "/usr/local/bin"
subprocess.run(["sh", str(installer)], check=True, env=env)
uv_version = subprocess.run(
    ["uv", "--version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(uv_version)
if not uv_version.startswith("uv 0.12.23 "):
    raise RuntimeError(f"unexpected uv version: {uv_version}")

In [ ]:
import shutil
from pathlib import PurePosixPath
from zipfile import ZipFile

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("upload exactly one repository ZIP")

archive = Path(next(iter(uploaded)))
repo = Path("/content/smallpotato")
if repo.exists():
    shutil.rmtree(repo)

with ZipFile(archive) as zip_file:
    roots = {
        PurePosixPath(name).parts[0]
        for name in zip_file.namelist()
        if name and not name.startswith("__MACOSX/")
    }
    if roots != {"smallpotato"}:
        raise RuntimeError("repository ZIP must contain smallpotato/")
    zip_file.extractall("/content")

shutil.rmtree(repo / ".venv", ignore_errors=True)
shutil.rmtree(
    repo / "reproductions" / "rtmdet" / "env" / ".venv",
    ignore_errors=True,
)
os.chdir(repo)
print(repo)

In [ ]:
subprocess.run(["uv", "sync", "--locked"], cwd=repo, check=True)

In [ ]:
process = subprocess.Popen(
    [
        "uv",
        "run",
        "python",
        "-m",
        "reproductions.rtmdet.reproduce",
    ],
    cwd=repo,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
print(f"exit status {returncode}")

In [ ]:
import json

runs = repo / "runs" / "rtmdet"
run_dir = max(
    (path for path in runs.iterdir() if path.is_dir()),
    key=lambda path: path.name,
)
summary = json.loads((run_dir / "summary.json").read_text())
print(run_dir)
print(json.dumps(summary, indent=2))

In [ ]:
archive_base = Path("/content") / f"rtmdet-{run_dir.name}"
run_archive = shutil.make_archive(
    str(archive_base),
    "gztar",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
print(run_archive)
files.download(run_archive)